##  03 Find Project Root

## Goal

- Sometimes notebooks are executed from different folders.
- To build reliable paths, we can locate the **project root** automatically by searching upward for a marker such as `.git`.

## Idea

- If your project contains a `.git` folder, then that folder is a good marker for the project root.
- We can start at the current working directory and move upward until we find it.

In [1]:
from pathlib import Path

def find_project_root(marker=".git"):
    path = Path.cwd()

    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent

    raise RuntimeError(f"Project root not found. Could not locate marker: {marker}")

In [2]:
try:
    root = find_project_root()
    print("Project root found:")
    print(root)

    data_path = root / "data" / "countries.geojson"
    print("\nExample data path from root:")
    print(data_path)
    print("Exists:", data_path.exists())
except RuntimeError as e:
    print("Notice:")
    print(e)

Project root found:
/Users/esther/Documents/4543--spatial_data_and_maps

Example data path from root:
/Users/esther/Documents/4543--spatial_data_and_maps/data/countries.geojson
Exists: True


## Why This Matters

- A root-finding function makes notebooks more reliable because the path logic does not depend as heavily on where the notebook happened to start.

- This becomes especially useful once students begin using:

  - `src/` layouts
  - helper libraries
  - nested notebook folders
  - bigger projects

In [3]:
# Optional: inspect the current directory and its parents
current = Path.cwd()
print("Current directory and parents:")
for p in [current, *current.parents]:
    print(" -", p)

Current directory and parents:
 - /Users/esther/Documents/4543--spatial_data_and_maps/Assignments/02-Missile_Geometry_202/_micro_lessons/00-Paths
 - /Users/esther/Documents/4543--spatial_data_and_maps/Assignments/02-Missile_Geometry_202/_micro_lessons
 - /Users/esther/Documents/4543--spatial_data_and_maps/Assignments/02-Missile_Geometry_202
 - /Users/esther/Documents/4543--spatial_data_and_maps/Assignments
 - /Users/esther/Documents/4543--spatial_data_and_maps
 - /Users/esther/Documents
 - /Users/esther
 - /Users
 - /


## Exercise A

Answer without writing code:

1. What happens when `find_project_root()` is called in a project with no `.git` folder? If the method cannot discover a .git folder above the current directory, it just continues upward until it hits the top of the filesystem. Because it never detects a marker, it returns to the default location rather than failing. In other words, it discreetly declares that no project root can be identified.
   
2. Name two other files or folders that could serve as a reliable project root marker.A project's root is typically identified by additional top-level files. Something like a pyproject.toml or requirements.txt files function well since they are almost always at the root of a Python project and do not move around.
   
3. Why is locating the root this way more robust than assuming the notebook always runs from the same folder? Notebooks can be opened from a variety of settings, each with their own working directory. Your code becomes much more stable when you search upward for a known marker rather to relying on where the notebook occurs to run. It can adapt to different machines, editors, and directory structures without breaking.

## Exercise B

Call `find_project_root()` with a different marker.

1. Try `"pyproject.toml"` — does it find a root? What does that tell you?
2. Try a marker that definitely doesn't exist (e.g. `"banana"`) — what happens, and why is the `RuntimeError` message useful?

In [4]:
from pathlib import Path

def find_project_root(marker):
    """Walk upward until the marker is found, or raise an error."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError(f"Could not find project root using marker: {marker}")

# 1. Try using "pyproject.toml"
try:
    root_pyproject = find_project_root("pyproject.toml")
    print("Found project root using pyproject.toml:", root_pyproject)
except RuntimeError as e:
    print("pyproject.toml test:", e)

# 2. Try using a marker that definitely doesn't exist
try:
    root_banana = find_project_root("banana")
    print("Found project root using banana:", root_banana)
except RuntimeError as e:
    print("banana test:", e)

Found project root using pyproject.toml: /Users/esther/Documents/4543--spatial_data_and_maps/Assignments/02-Missile_Geometry_202
banana test: Could not find project root using marker: banana


## Exercise C

Use `find_project_root()` to build a reliable path to this module's `countries.geojson` data file.

Use the directory tree printed above to figure out the correct subfolders from root, then replace `"???"` in the cell below.

In [6]:
from pathlib import Path

root = find_project_root(".git")

# Build the path from root to this module's data folder and check countries.geojson
# Hint: look at the directory tree printed above to figure out the right subfolders
data_file = root / "data" / "countries.geojson"

print("Looking for:", data_file)
print("Exists:", data_file.exists())

Looking for: /Users/esther/Documents/4543--spatial_data_and_maps/data/countries.geojson
Exists: True


## Optional Advanced — Multiple Markers and Custom Start

Extend `find_project_root` to accept:

- `markers` — a list of marker names; stop at the first directory that contains **any** of them
- `start` — an optional starting path (defaults to `Path.cwd()` if not given)

This lets you test the function from any location and makes it more flexible for projects that don't use git.

In [ ]:
from pathlib import Path

def find_project_root_multi(markers=(".git", "pyproject.toml", "setup.py"), start: Path = None):
    """Return the first ancestor directory that contains any of the marker files/folders."""
    # Your code here
    pass

root = find_project_root_multi()
print("Root:", root)

## Check Your Understanding

1. `find_project_root()` walks upward until `path == path.parent`. What does that condition mean — when does it become true? That condition becomes true only when you’ve reached the very top of the filesystem — the root directory. At that point, moving “up” no longer changes the path, so the directory is its own parent. It’s the signal that there’s nowhere left to climb.

2. You call `find_project_root()` from a notebook and get back `/project`. You then write `root / "data" / "cities.json"`. What is the full absolute path that produces? If find_project_root() returns /project, then adding "data" and "cities.json" builds the absolute path /project/data/cities.json. It simply follows the folder structure downward from the root you discovered.

3. A teammate hardcodes `Path("/Users/them/project/data/cities.json")` instead of using `find_project_root()`. What breaks when you run their notebook? Their notebook works only on their machine because that absolute path includes their username and their personal folder layout. When you run it, your computer doesn’t have /Users/them/..., so the file lookup fails immediately. Using find_project_root() avoids that brittleness by adapting to each user’s environment.

---
# Summary

## Students should now understand

- what the working directory is
- how relative and absolute paths differ
- how to build paths into other folders
- how to check whether files exist
- how to locate a project root marker

## Recommended next step

Move into:

`01_JSON_GeoJSON`

because now students are ready to locate files **before** opening and inspecting them.

That tiny detail saves a lot of chaos later. A shocking amount, really.